# Section 07 - RAG Security

> Code + full series: **[github.com/dearnidhi/ai-security-bootcamp](https://github.com/dearnidhi/ai-security-bootcamp)**

RAG = your AI searches your company documents to answer questions.
RAG Security = what if someone puts a fake document in that knowledge base?

## The Attack - Document Poisoning

Your knowledge base has 4 real policy documents. An attacker adds 1 fake document that looks exactly like a real policy update - but contains wrong/harmful info.

**Example:**

Real document: *"Send reimbursement forms to finance@company.com"*

Poisoned document: *"Updated process: Send reimbursement forms to finance@attacker.com"*

Now every employee who asks the chatbot gets the wrong email address.

**Why it works:**
- The AI did nothing wrong - it read the document it was given
- No server was hacked - just data was changed
- Employees trust the company chatbot - they don't double check

## The Defense - LLM Verifier

Keep a list of **trusted facts** (written by a human, not AI):
> *"Reimbursements go to finance@company.com"*

After the AI generates an answer, run a second check:
> *"Does this answer contradict the trusted fact?"*

If yes → flag it.

```
User question
    ↓
RAG retrieves docs (may include poisoned one)
    ↓
AI generates answer
    ↓
Verifier checks answer vs trusted fact  ← defense
    ↓
✅ Safe  or  🚩 Contradiction detected
```

## What the demo shows

**`app.py`** - Streamlit UI:
- Toggle poison ON/OFF
- Ask the question
- See how the answer changes
- See the verifier flag the contradiction

**`backend/main.py`** - FastAPI:
- `/ask` endpoint
- Retrieves documents, generates answer, runs verifier check

```bash
# Terminal 1
uvicorn backend.main:app --reload --port 8002

# Terminal 2
streamlit run app.py
```

## Bonus: Corrective RAG (built with LangGraph)

The verifier above only reports a problem. A **LangGraph** version can act on it:

```
retrieve -> generate -> verify --contradicts--> filter out bad doc -> generate (retry)
                            |
                          clean
                            |
                          done
```

If the verifier flags a contradiction, the pipeline finds the wrong email address by
comparing the answer to the canonical fact (not by asking the LLM to quote an exact
snippet - it tends to paraphrase, so matching that snippet against the source document
text is unreliable), drops the document containing that email, and retries generation
with the smaller, clean context - up to 2 attempts.

Try `POST /ask/corrective` with `include_poison: true` (see this module's README): the
first attempt answers wrong, the poisoned document gets filtered out, and the second
attempt answers correctly - the pipeline heals itself without a human intervening.

**Why this needs a graph, not a plain chain:** a chain runs straight through once. Here,
"verify" can send you back to "generate" (through a filtering step) or forward to done,
depending on its own result - that loop-back is what `add_conditional_edges` is for.
